# Training on soft labels: typed-decisions

[LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) has
1,200 training and 400 test records from four operational workflows: agent-trace monitoring,
customer service, invoice processing and security incidents. Each record is a structured JSON
state with the same five questions for its workflow.

Its labels are **soft**: each question's answer is a probability spread from several raters,
for example `{"continue": 0.28, "human_review": 0.43, "observe": 0.25, "stop": 0.03}`, rather
than a single right answer. sysone trains on these distributions directly (passed as `gold`).

Because raters often disagree, the notebook measures three things on held-out records:
- **accuracy**: does the model pick the raters' top answer?
- **distance**: how far the predicted probabilities are from the raters' spread (total variation
  distance, 0 = identical, 1 = completely different);
- **score error**: for score questions, the gap between predicted and rated expected level.

It trains Qwen3-0.6B with `tuning="head"` and takes about 10 minutes on a Mac.

In [ ]:
import json
import time
from collections import defaultdict

import pandas as pd
from datasets import load_dataset

from sysone import DecoderModel

## Load the data

`state`, `questions` and `gold` are JSON text; sysone decodes them itself. The workflow is used as
the row's `category`.

In [ ]:
def load_rows(split):
    ds = load_dataset("LocalLLaMA/typed-decisions", "all", split=split)
    return [{"state": r["state"], "questions": json.loads(r["questions"]), "gold": json.loads(r["gold"]),
             "category": r["workflow"]} for r in ds]


train = load_rows("train")
test = load_rows("test")
print("%d training records, %d test records" % (len(train), len(test)))

example = train[0]
print("\nworkflow:", example["category"])
print("state:", example["state"][:300], "...")
print("\nquestion 'action':", example["questions"]["action"]["instructions"])
print("rater probabilities:", example["gold"]["action"]["probabilities"])

## Scoring

Every answer is turned into a probability distribution over the question's options and compared
with the raters' distribution. A noul's options are `false` / `true`, and a score's are its level
indices `"0"`, `"1"`, ...

In [ ]:
TYPES = ("choice", "score", "noul")


def predicted_distribution(answer):
    if answer["type"] == "noul":
        return {"false": 1 - answer["noul"], "true": answer["noul"]}
    return answer["probabilities"]


def compare(pred, gold):
    """(picked the raters' top answer, total variation distance, |expected level difference| or None)"""
    keys = list(gold)
    top_hit = max(keys, key=lambda k: pred.get(k, 0.0)) == max(keys, key=gold.get)
    distance = 0.5 * sum(abs(pred.get(k, 0.0) - gold[k]) for k in keys)
    level_gap = None
    if all(k.isdigit() for k in keys):  # a score question
        level_gap = abs(sum(int(k) * pred.get(k, 0.0) for k in keys) - sum(int(k) * gold[k] for k in keys))
    return top_hit, distance, level_gap


def summarize(per_question):
    """per_question: list of (type, category, top_hit, distance, level_gap)"""
    out = {}
    for t in TYPES:
        rows = [q for q in per_question if q[0] == t]
        out["accuracy " + t] = sum(q[2] for q in rows) / len(rows)
    out["accuracy overall"] = sum(q[2] for q in per_question) / len(per_question)
    out["distance"] = sum(q[3] for q in per_question) / len(per_question)
    gaps = [q[4] for q in per_question if q[4] is not None]
    out["score error"] = sum(gaps) / len(gaps)
    return out


def evaluate(model, rows):
    preds = model.predict([{"state": r["state"], "questions": r["questions"]} for r in rows])
    results = []
    for row, pred in zip(rows, preds):
        for qid, answer in pred["answers"].items():
            gold = row["gold"][qid]["probabilities"]
            results.append((answer["type"], row["category"], *compare(predicted_distribution(answer), gold)))
    return results

## Baselines

- **random guess**: every option equally likely.
- **average answer**: for each workflow question, the raters' average distribution over the
  training set. Every record in a workflow has the same questions, so this is a strong bar; a
  model that beats it is reading each record, not just learning the typical answer.

In [ ]:
def baseline(rows, distribution_for):
    results = []
    for row in rows:
        for qid, q in row["questions"].items():
            gold = row["gold"][qid]["probabilities"]
            results.append((q["type"], row["category"], *compare(distribution_for(row, qid, gold), gold)))
    return results


def uniform(row, qid, gold):
    return {k: 1 / len(gold) for k in gold}


totals = defaultdict(lambda: defaultdict(float))
counts = defaultdict(int)
for row in train:
    for qid in row["questions"]:
        key = (row["category"], qid)
        counts[key] += 1
        for option, p in row["gold"][qid]["probabilities"].items():
            totals[key][option] += p


def average_answer(row, qid, gold):
    key = (row["category"], qid)
    return {k: totals[key][k] / counts[key] for k in gold}


results = {
    "random guess": summarize(baseline(test, uniform)),
    "average answer": summarize(baseline(test, average_answer)),
}
pd.DataFrame(results).T.round(3)

## Before training

sysone won't predict with an untrained head, so we fit one epoch with a learning rate of 0 on a
few records. The head keeps its random initial weights.

In [ ]:
untrained = DecoderModel(tuning="head", epochs=1, head_lr=0.0, verbose=0).fit(train[:20])
results["before training"] = summarize(evaluate(untrained, test))
pd.DataFrame(results).T.round(3)

## Train

The soft `gold` distributions are the training targets. 10% of records are held back for
validation and early stopping, split evenly across workflows.

In [ ]:
start = time.time()
model = DecoderModel(tuning="head")
model.fit(train)
print("trained in %.1f minutes" % ((time.time() - start) / 60))
model.report_["validation"]

## After training

Lower is better for **distance** and **score error**; higher is better for accuracy.

In [ ]:
after = evaluate(model, test)
results["after training"] = summarize(after)
pd.DataFrame(results).T.round(3)

### By workflow

In [ ]:
by_workflow = {}
for workflow in sorted({q[1] for q in after}):
    rows = [q for q in after if q[1] == workflow]
    by_workflow[workflow] = {
        "accuracy": sum(q[2] for q in rows) / len(rows),
        "distance": sum(q[3] for q in rows) / len(rows),
        "questions": len(rows),
    }
pd.DataFrame(by_workflow).T.round(3)

## Reading the results

- Raters often disagree, so accuracy has a ceiling well below 100%: when the raters split 45/55,
  either answer is defensible.
- **average answer** is the bar to beat. With a frozen backbone and only 1,080 training records
  the head may not get far past it; that is what fine-tuning the backbone (`tuning="lora"` /
  `"qlora"`, coming next) is for.
- Note that probabilities are not calibrated yet (temperatures stay at 1.0), which affects
  **distance** more than accuracy.